# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [20]:
!pip uninstall -y langchain langchain-core langchain-openai langchain-community langchain-experimental
!pip install langchain langchain-core langchain-openai langchain-community langchain-text-splitters langchain-huggingface

Found existing installation: langchain 1.2.10
Uninstalling langchain-1.2.10:
  Successfully uninstalled langchain-1.2.10
Found existing installation: langchain-core 1.2.17
Uninstalling langchain-core-1.2.17:
  Successfully uninstalled langchain-core-1.2.17
Found existing installation: langchain-openai 1.1.10
Uninstalling langchain-openai-1.1.10:
  Successfully uninstalled langchain-openai-1.1.10
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
  Using cached langchain-1.2.10-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain_core-1.2.17-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_openai-1.1.10-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
Using cached langchain-1.2.10-py3-none-any.whl (111 kB)
Using cached langchain_core-1.2.17-py3-none-any.whl (502 kB)
Using cached langchain_openai-1.1.10-py3-none-a

In [4]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [5]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

### Example 1

#### Create our QandA application

In [5]:
# 1. Full clean uninstall (includes corrupted packages)
!pip uninstall langchain langchain-core langchain-community langchain-openai langchain-huggingface langchain-text-splitters langchain-text-splitters-* -y

# 2. Clear pip cache
!pip cache purge

# 3. Install compatible pinned versions
!pip install "langchain-core>=0.2.28,<0.3" langchain-openai langchain-huggingface "langchain-community>=0.2.16,<0.3" langchain-text-splitters

# 4. Verify
!pip list | grep langchain

ERROR: Invalid requirement: 'langchain-text-splitters-*': Expected end or semicolon (after name and no valid version specifier)
    langchain-text-splitters-*
                            ^
Files removed: 538
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-huggingface to dete

langchain                                0.2.17
langchain-classic                        1.0.1
langchain-community                      0.2.19
langchain-core                           0.2.43
langchain-huggingface                    0.0.3
langchain-openai                         0.1.25
langchain-text-splitters                 0.2.4


In [6]:
import sys
!{sys.executable} -m pip list | grep -i langchain

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/pip/__main__.py", line 22, in <module>
    from pip._internal.cli.main import main as _main
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/build_env.py", line 19, in <module>
^C


In [3]:
!pip install -U langchain

In [1]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain  # ✅ FIXED
from langchain_core.output_parsers import StrOutputParser, BaseOutputParser
from langchain_core.documents import Document
from pydantic import BaseModel, Field
from langchain.evaluation.qa import QAGenerateChain

In [ ]:
#ONly to use with csv files
#file = r'C:\Users\Denish\Desktop\TA\Week_8_Lab_Soluiton\Day_2\OutdoorClothingCatalog_1000(2).csv'
#loader = CSVLoader(file_path=file)
#data = loader.load()

In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

docs = text_splitter.split_documents(docs)

# Create embeddings and vectorstore
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

vectorstore = InMemoryVectorStore.from_documents(
    docs, embeddings
)

retriever = vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain.chains.combine_documents import create_stuff_documents_chain

llm = ChatOpenAI(temperature=0.0)

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

CONTEXT:
{context}

<<<<>>>>>

QUESTION: {question}

ANSWER:
""")

qa = create_stuff_documents_chain(llm, prompt, document_separator="<<<<>>>>>")

# Fixed chain - provides both context AND question
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | qa
)


result = chain.invoke("Do the Cozy Comfort Pullover Set have side pockets?")
print(result)

#### Hard-coded examples

In [ ]:
!pip install langchain_classic

In [ ]:
!pip uninstall -y langchain langchain-core langchain-openai pydantic
!pip install langchain langchain-core langchain-openai pydantic --upgrade

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
             "   Answer: Yes\n"
             "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
             "   Answer: The DownTek collection\n"
             "Query: {query}\n"
             "Answer:"
)


class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)


llm = ChatOpenAI(model="gpt-3.5-turbo")
llm_chain = prompt_template | llm | AnswerOutputParser()

query = "Is the Cozy Comfort Pullover Set available in different colors?"
result = llm_chain.invoke({"query": query})
print(result.answer)

In [ ]:
# Run this to see what's in langchain.evaluation
import langchain.evaluation.qa
print(dir(langchain.evaluation.qa))

#### LLM-Generated examples

In [8]:
from langchain.evaluation.qa import QAGenerateChain
from langchain.chains import LLMChain
data = docs

example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(model="gpt-3.5-turbo"))

# Your EXACT workflow now works!
llm_chain = LLMChain(llm=llm, prompt=prompt_template, output_parser=AnswerOutputParser())

new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

d_flattened = [data['qa_pairs'] for data in new_examples]


ModuleNotFoundError: No module named 'langchain.evaluation'

#### Combine examples

In [ ]:
# examples += new_example
examples += d_flattened

In [ ]:
examples

In [ ]:
result = qa.invoke({
    "question": examples[6]["query"],  # ← "question" not "input"
    "context": retriever.invoke(examples[6]["query"])
})
print(result)

### Manual Evaluation - Fun part

In [ ]:
import langchain
langchain.debug = True

# qa needs BOTH question AND context
result = qa.invoke({
    "question": examples[0]["query"],
    "context": retriever.invoke(examples[0]["query"])
})
print(result)

In [ ]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [10]:
# predictions must be list of dicts like:
# [{'query': '...', 'answer': '...', 'result': '...'}, ...]

predictions = []
for i, eg in enumerate(examples[:5]):  # Limit for testing
    docs = retriever.invoke(eg["query"])
    result = qa.invoke({"question": eg["query"], "context": docs})

    pred_dict = {
        'query': eg["query"],
        'answer': eg["answer"],
        'result': result
    }
    predictions.append(pred_dict)

# Now your print loop works perfectly!
for i, eg in enumerate(examples[:5]):
    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real Answer:", predictions[i]['answer'])
    print("Predicted Answer:", predictions[i]['result'])
    print()

NameError: name 'examples' is not defined

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=1)

# Simple eval prompt (no classes)
eval_prompt = PromptTemplate(
    input_variables=["query", "answer", "result"],
    template="""Score PREDICTED vs ANSWER for QUERY (0-1, higher=better match).

Query: {query}
Answer: {answer}
Predicted: {result}

Score:"""
)

eval_chain = eval_prompt | llm | StrOutputParser()  # Simple string parser

# Your EXACT workflow (no classes)
graded_outputs = []
for i, eg in enumerate(predictions):
    score_raw = eval_chain.invoke({
        "query": predictions[i]['query'],
        "answer": predictions[i]['answer'],
        "result": predictions[i]['result']
    })
    score = float(score_raw.strip())  # Extract number

    graded_outputs.append({'score': score})

    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real:", predictions[i]['answer'])
    print("Predicted:", predictions[i]['result'])
    print("Score:", score, "\n")

In [ ]:
graded_outputs

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [6]:
# Here we use the ChatOpenAI LLM to generate answers based on retrieved context
from langchain_community.document_loaders import TextLoader
from langchain_core.runnables import RunnablePassthrough

# Load the text file
loader = TextLoader("/content/Alice_Data.txt")
docs = loader.load()
# Split documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter.split_documents(docs)

# Create vectorstore
vectorstore = InMemoryVectorStore.from_documents(
    split_docs, HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)
retriever = vectorstore.as_retriever()

# Create QA chain
llm = ChatOpenAI(temperature=0)
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:

CONTEXT:
{context}

QUESTION: {question}

ANSWER:
""")

qa_chain = create_stuff_documents_chain(llm, prompt)

qa_chain_with_docs = (
    {"context": retriever, "question": RunnablePassthrough()}
    | qa_chain
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# testing it out

question = "Who is Lady Adler?"
result = qa_chain_with_docs.invoke(question)
print(result)

In [ ]:
question = "What is SHerlock#s preofession?"
result = qa_chain_with_docs.invoke(question)
print(result)

In [ ]:
question = "Does Mr Holmes have a brother?"
result = qa_chain_with_docs.invoke(question)
print(result)

In [ ]:
# testing it out

question = "Who is Alice?"
result = qa_chain_with_docs.invoke(question)
print(result)

In [ ]:
question = "Does Alice represent well the avarage western young girl?"
result = qa_chain_with_docs.invoke(question)
print(result)

In [ ]:
question = "Why does Alice get lost?"
result = qa_chain_with_docs.invoke(question)
print(result)

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [ ]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [ ]:
examples

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain).
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

In [11]:
!pip install ragas

In [11]:
# ---- build everything needed for ragas ----

# if docs already exist from Alice loader this line just reuses them
data = docs

# create example generator
from langchain.evaluation.qa import QAGenerateChain
from langchain_openai import ChatOpenAI

example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI())

# generate QA examples
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

# flatten examples
d_flattened = [d["qa_pairs"] for d in new_examples]
examples = [item for sublist in d_flattened for item in sublist]

/usr/local/lib/python3.12/dist-packages/langchain/chains/llm.py:366: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


In [15]:
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset

all_predictions = []

for example in examples:
    q = example
    d = retriever.invoke(q)
    a = qa_chain.invoke({"question": q, "context": d})

    all_predictions.append({
        "question": q,
        "answer": a,
        "contexts": [doc.page_content for doc in d],
        "ground_truth": ""
    })

dataset_all = Dataset.from_list(all_predictions)

print("evaluating faithfulness...")
f = evaluate(dataset_all, metrics=[faithfulness])
print(f)

/tmp/ipykernel_38946/1068336239.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
/tmp/ipykernel_38946/1068336239.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall


evaluating faithfulness...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

{'faithfulness': 1.0000}


1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

**Faithfulness**

High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [16]:
print("evaluating context recall...")
r = evaluate(dataset_all, metrics=[context_recall])
print(r)

evaluating context recall...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

{'context_recall': nan}


In [19]:
from ragas import evaluate
from ragas.metrics import faithfulness
from datasets import Dataset

# Use question 2 (economic significance) – produces long detailed answer
high_data = Dataset.from_list([all_predictions[2]])
high_score = evaluate(high_data, [faithfulness])["faithfulness"]
print("HIGH Faithfulness (real answer):", high_score)

# LOW – contradicting claims
fake = all_predictions[2].copy()
fake["answer"] = "New York City has no economic importance. It is a small rural town with no financial institutions."

low_data = Dataset.from_list([fake])
low_score = evaluate(low_data, [faithfulness])["faithfulness"]
print("LOW Faithfulness (fake answer):", low_score)

print("\n--- Explanation ---")
print(f"Real answer score: {high_score[0]:.2f} <- claims supported by context")
print(f"Fake answer score: {low_score[0]:.2f} <- claims contradict context")

/tmp/ipykernel_38946/914047098.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

HIGH Faithfulness (real answer): [1.0]


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

LOW Faithfulness (fake answer): [0.0]

--- Explanation ---
Real answer score: 1.00 <- claims supported by context
Fake answer score: 0.00 <- claims contradict context


In [20]:
print("Question:", all_predictions[2]["question"])
print("Answer:", all_predictions[2]["answer"])

Question: query
Answer: The Caterpillar asked Alice, "Who are you?"


**Context Relevancy**

High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [24]:
from ragas import evaluate
from ragas.metrics import context_recall
from datasets import Dataset
from langchain_openai import ChatOpenAI

evaluator_llm = ChatOpenAI()

# HIGH – real docs
high_data = Dataset.from_list([all_predictions[2]])
high_score = evaluate(high_data, metrics=[context_recall], llm=evaluator_llm)["context_recall"]
print("HIGH Context Recall (real docs):", high_score)

# LOW – off-topic docs
fake = all_predictions[2].copy()
fake["contexts"] = [
    "The Eiffel Tower is located in Paris, France.",
    "Python is a popular programming language used in data science."
]

low_data = Dataset.from_list([fake])
low_score = evaluate(low_data, metrics=[context_recall], llm=evaluator_llm)["context_recall"]
print("LOW Context Recall (fake docs):", low_score)

/tmp/ipykernel_38946/4168301308.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_recall


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

HIGH Context Recall (real docs): [1.0]


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

LOW Context Recall (fake docs): [nan]


In [27]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from datasets import Dataset

# helper
def get_contexts_str(contexts):
    return [
        doc.page_content if hasattr(doc, "page_content") else str(doc)
        for doc in contexts
    ] if contexts and hasattr(contexts[0], "page_content") else contexts


def evaluate_answer(question, answer, contexts, ground_truth=None, test_name=""):
    contexts_str = get_contexts_str(contexts)

    data = {
        "question": [question],
        "answer": [answer],
        "contexts": [contexts_str],
        "ground_truth": [ground_truth if ground_truth else answer],
    }

    dataset = Dataset.from_dict(data)
    results = evaluate(dataset=dataset, metrics=[faithfulness, answer_relevancy, context_recall])

    print(f"\n{'='*60}")
    print(f"TEST: {test_name}")
    print(f"Faithfulness:     {results['faithfulness'][0]:.4f}")
    print(f"Answer Relevancy: {results['answer_relevancy'][0]:.4f}")
    print(f"Context Recall:   {results['context_recall'][0]:.4f}")

    return results


# Test it
evaluate_answer(
    question=all_predictions[2]["question"],
    answer=all_predictions[2]["answer"],
    contexts=all_predictions[2]["contexts"],
    ground_truth=all_predictions[2]["ground_truth"],
    test_name="Economic Significance",
)

/tmp/ipykernel_38946/1099233097.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_recall
/tmp/ipykernel_38946/1099233097.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_recall
/tmp/ipykernel_38946/1099233097.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, answer_relevancy, context_recall


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[1]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')



TEST: Economic Significance
Faithfulness:     1.0000
Answer Relevancy: nan
Context Recall:   1.0000


{'faithfulness': 1.0000, 'answer_relevancy': nan, 'context_recall': 1.0000}

2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [28]:
from ragas.metrics import faithfulness, answer_relevancy, context_recall

dataset_all = Dataset.from_list(all_predictions)

results = evaluate(
    dataset_all,
    metrics=[faithfulness, answer_relevancy, context_recall]
)

print(results)
results.to_pandas()

/tmp/ipykernel_38946/3488777761.py:1: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_recall
/tmp/ipykernel_38946/3488777761.py:1: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_recall
/tmp/ipykernel_38946/3488777761.py:1: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, answer_relevancy, context_recall


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[1]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[4]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[7]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[10]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[13]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[16]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[19]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[22]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executo

{'faithfulness': 1.0000, 'answer_relevancy': nan, 'context_recall': nan}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_recall
0,query,"[Down, down, down. Would the fall _never_ come...","The question is ""Who are you?""",,1.0,NaN,NaN
1,answer,[“Have you guessed the riddle yet?” the Hatter...,The Hatter and the March Hare do not know the ...,,1.0,NaN,NaN
2,query,"[Down, down, down. Would the fall _never_ come...","The Caterpillar asked Alice, ""Who are you?""",,1.0,NaN,NaN
3,answer,[“Have you guessed the riddle yet?” the Hatter...,The Hatter and the March Hare do not know the ...,,1.0,NaN,NaN
4,query,"[Down, down, down. Would the fall _never_ come...","The question is ""Who are you?""",,1.0,NaN,NaN
5,answer,[“Have you guessed the riddle yet?” the Hatter...,The Hatter and the March Hare do not know the ...,,1.0,NaN,NaN
6,query,"[Down, down, down. Would the fall _never_ come...","The question is ""Who are you?""",,1.0,NaN,NaN
7,answer,[“Have you guessed the riddle yet?” the Hatter...,The Hatter and the March Hare do not know the ...,,1.0,NaN,NaN
8,query,"[Down, down, down. Would the fall _never_ come...","The Caterpillar asked Alice, ""Who are you?""",,1.0,NaN,NaN
9,answer,[“Have you guessed the riddle yet?” the Hatter...,The Hatter and the March Hare do not know the ...,,1.0,NaN,NaN


In [33]:
# run the queries as a batch for efficiency
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset

# Step 1 – batch predictions
all_preds = []
for example in examples:
    q = example
    d = retriever.invoke(q)
    a = qa_chain.invoke({"question": q, "context": d})
    all_preds.append({
        "question": q,
        "answer": a,
        "contexts": [doc.page_content for doc in d],
        "ground_truth": ""
    })

dataset_all = Dataset.from_list(all_preds)

print("evaluating faithfulness...")
r = evaluate(dataset_all, metrics=[faithfulness])
print(r)

print("evaluating context recall...")
r = evaluate(dataset_all, metrics=[context_recall])
print(r)

/tmp/ipykernel_38946/2610205824.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
/tmp/ipykernel_38946/2610205824.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall


evaluating faithfulness...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

{'faithfulness': 1.0000}
evaluating context recall...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

{'context_recall': nan}
